In [ ]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

In [ ]:
up = files.upload()
nama_file = list(up.keys())[0]

print("File terbaca:", nama_file)

In [ ]:
df = pd.read_excel(nama_file)

JUMLAH_KOLOM_DEMOGRAFI = next(
    (i for i, kolom in enumerate(df.columns)
     if re.search(r"\b[A-Za-z]{1,4}\d+\b", str(kolom))),
    None
)

if JUMLAH_KOLOM_DEMOGRAFI is None:
    raise ValueError("Kolom pertanyaan tidak terdeteksi.")

butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))

In [ ]:
skala = {
  "Sangat Tidak Setuju (STS)": 1,
  "Tidak Setuju (TS)": 2,
  "Netral (N)": 3,
  "Setuju (S)": 4,
  "Sangat Setuju (SS)": 5,
}

num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")

print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
num.head()

In [ ]:
dimensi = {}

for kolom in num.columns:
    kode = re.search(r"\b([A-Za-z]{1,4})\d+\b", str(kolom))

    if kode:
        nama = kode.group(1).upper()
        dimensi.setdefault(nama, []).append(kolom)

if not dimensi:
    raise ValueError("Kode dimensi tidak ditemukan.")

if num.isna().any().any():
    raise ValueError("Ada jawaban kosong/tidak dikenali.")

for nama, daftar in dimensi.items():
    print(f"Dimensi {nama}: {len(daftar)} butir")

In [ ]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)

print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []

for nama, daftar in dimensi.items():
  x = num[daftar]
  total = x.sum(axis=1)

  for i, kolom in enumerate(daftar, 1):
    r, p = stats.pearsonr(x[kolom], total)
    r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])

    hasil_validitas.append({
        "Dimensi": nama,
        "Butir": f"B{i}",
        "r hitung": round(r, 3),
        "Corrected r": round(r_koreksi, 3),
        "p-value": round(p, 4),
        "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
    })

tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

In [ ]:
def cronbach_alpha(x):
  k = x.shape[1]
  return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
  return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
    else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
  x = num[daftar]
  a = cronbach_alpha(x)
  hasil_reliabilitas.append({
    "Dimensi": nama,
    "Jumlah butir": len(daftar),
    "Cronbach's alpha": round(a, 3),
    "Kategori": kategori(a),
  })

  for i, kolom in enumerate(daftar, 1):
    if len(daftar) > 2:
      alpha_dihapus.append({
        "Dimensi": nama,
        "Butir": f"B{i}",
        "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3),
      })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)

display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

In [ ]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")

with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
  gabung.to_excel(w, sheet_name="Validitas", index=False)
  tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)

files.download("hasil_uji_validitas_reliabilitas.xlsx")